# <font color="steelblue">Masas mamográficas</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a>

No olvides hacer una copia si deseas utilizarlo.


## <font color="steelblue">Objetivos del proyecto</font>

A partir de **atributos morfológicos BI-RADS** y la **edad** de la paciente, construir, comparar y **desplegar** un sistema de **diagnóstico asistido (CAD)** que prediga si una masa mamográfica es **benigna o maligna**, con el fin de **reducir biopsias innecesarias** sin dejar de detectar los tumores malignos. Lo que hace especial a este proyecto es su marco **clínico y de coste**:

* La masa se decide **sin biopsia**, así que el **coste de los errores es asimétrico**: un falso negativo (maligno no detectado) es mucho más grave que un falso positivo (biopsia de más).
* Hay una variable, **BI-RADS assessment**, que **no debéis usar como predictora** (sería copiar el juicio del radiólogo), pero que sí podéis usar como **referencia clínica** para comparar vuestro modelo.
* El valor de un CAD clínico depende de que su **decisión sea interpretable**.

Al terminar, debéis ser capaces de **codificar correctamente** variables nominales y ordinales, **evitar la fuga** de la valoración BI-RADS, **optimizar el umbral según el coste clínico**, **comparar el modelo con el criterio del radiólogo**, interpretarlo y desplegarlo.


## <font color="steelblue">El conjunto de datos</font>

### <font color="steelblue">Origen y estructura</font>

El conjunto recoge **961 masas** detectadas en mamografías digitales de campo completo, obtenidas en el **Instituto de Radiología de la Universidad de Erlangen-Núremberg entre 2003 y 2006**. De ellas, **516 resultaron benignas y 445 malignas**, un reparto **bastante equilibrado** que evita los problemas habituales de desbalanceo.

Para leer las variables hay que conocer el sistema **BI-RADS** (*Breast Imaging Reporting and Data System*), el estándar del Colegio Americano de Radiología que normaliza cómo se describe un hallazgo mamográfico. Ese sistema establece que toda masa se caracterice mediante tres descriptores —**forma**, **margen** y **densidad**— y que el radiólogo emita, a partir de ellos y del contexto clínico, una **valoración global** en una escala del 1 al 5. En este conjunto, esa valoración fue asignada mediante un **proceso de doble revisión** por parte de los facultativos.

Conviene retener una idea que condiciona todo el análisis: **las predictoras no son mediciones objetivas de un aparato, sino apreciaciones visuales de un radiólogo** codificadas en categorías.

### <font color="steelblue">Diccionario de variables</font>

| Variable | Tipo | Valores | Descripción |
|---|---|---|---|
| `BI-RADS assessment` | **Ordinal** | 1–5 | Valoración global del radiólogo: 1 = definitivamente benigna, 5 = muy sugestiva de malignidad. **⚠️ No predictiva: excluir de `X`.** Útil solo como **referencia de comparación**. |
| `Age` | Numérica (entero) | años | Edad de la paciente. La incidencia de malignidad **aumenta con la edad**; es la única variable genuinamente continua del conjunto. |
| `Shape` | **Nominal** | 1 = redonda · 2 = oval · 3 = lobulada · 4 = irregular | **Forma** de la masa. Las masas de contorno regular suelen corresponder a lesiones benignas; la **irregularidad** es un signo de sospecha, porque refleja un crecimiento desordenado que no respeta los planos del tejido. |
| `Margin` | **Nominal** | 1 = circunscrita · 2 = microlobulada · 3 = obscurecida · 4 = mal definida · 5 = espiculada | **Margen** o borde de la masa. Un margen **circunscrito** (nítido, bien delimitado) sugiere benignidad; el margen **espiculado** —con proyecciones radiales hacia el tejido circundante— es uno de los signos de malignidad más específicos. La categoría *obscurecida* indica que el borde queda **oculto** por tejido adyacente: no describe la lesión, sino una limitación de la observación. |
| `Density` | **Ordinal** | 1 = alta · 2 = isodensa · 3 = baja · 4 = grasa | **Densidad radiológica** de la masa comparada con el tejido mamario circundante. ⚠️ **La escala es decreciente:** a mayor número, **menor** densidad. Una masa de contenido **graso** es casi siempre benigna. |
| `Severity` | **Binaria** | 0 = benigna · 1 = maligna | **Variable objetivo**: el diagnóstico **histológico confirmado**, es decir, el resultado de la biopsia. Es una etiqueta de alta calidad, no una opinión. |

> **Ojo con la codificación:** `Shape` y `Margin` son **categóricas nominales**; sus números son **etiquetas**, no cantidades. Tratarlas como números (orden/distancia) es un error: usad **one-hot**. `Density` sí tiene un orden razonable.
>
> **Un matiz interesante:** la numeración de `Shape` y `Margin` está, en la práctica, **ordenada por grado de sospecha** (de redonda a irregular; de circunscrita a espiculada). Por eso tratarlas como numéricas «funciona» razonablemente bien y muchos trabajos lo hacen. Pero es una **coincidencia de la codificación**, no una propiedad de las variables: la categoría *obscurecida* (3), por ejemplo, rompe esa gradación, pues no describe la lesión sino la calidad de la imagen. El *one-hot* es la opción correcta, y comparar ambos tratamientos es un experimento instructivo.

### <font color="steelblue">Advertencias metodológicas</font>

1. **Por qué `BI-RADS` no es predictiva.** No es un dato objetivo sobre la lesión, sino **la conclusión del radiólogo** a partir de los otros tres descriptores. Usarla como predictora es **circular**: el modelo aprendería a imitar el juicio del médico, no a diagnosticar. Su función en el conjunto es otra y muy valiosa: sirve de **referencia de comparación**. Si se establece un umbral (por ejemplo, «se considera maligna toda masa con BI-RADS ≥ 4») se obtiene la **sensibilidad y la especificidad de los radiólogos**, y con ella un punto en el espacio ROC contra el que medir el modelo. La pregunta del proyecto no es «¿acierto mucho?», sino **«¿acierto más que el radiólogo, usando solo lo que él vio?»**.

2. **Cuidado con el sentido de `Density`.** Es la trampa de este conjunto: en `Shape` y `Margin` un número **alto** significa **más sospecha**, pero en `Density` un número alto significa **menos densidad** y, por tanto, **menos sospecha**. Al interpretar coeficientes o valores SHAP hay que tenerlo presente.

3. **Los valores faltantes del original.** El fichero de UCI contiene unos **167 valores ausentes** (codificados con `?`), muy desigualmente repartidos: alrededor de 76 en `Density`, 48 en `Margin`, 31 en `Shape` y 5 en `Age`. Eliminar las filas afectadas descarta cerca del **13 % de la muestra**, y hacerlo sin más **puede sesgar el resultado**: es plausible que un descriptor falte precisamente cuando la lesión era **difícil de caracterizar**, es decir, en los casos más interesantes. Si el fichero limpio ya ha decidido por vosotros, comprobad **qué decidió** y valorad las consecuencias.

4. **El fichero original contiene errores.** La columna `BI-RADS` presenta valores **fuera de rango** (se documentan valores como 55, imposibles en una escala de 1 a 5). Al excluirla del modelado el problema resulta inocuo, pero es un recordatorio de que **incluso los conjuntos canónicos contienen errores** y de que la exploración inicial no es un trámite.

5. **Descriptores subjetivos.** `Shape`, `Margin` y `Density` son apreciaciones visuales sujetas a **variabilidad entre observadores**: dos radiólogos pueden codificar la misma masa de forma distinta. El modelo hereda ese ruido, que fija un **techo de rendimiento** por debajo del 100 %. Una exactitud en torno al 80–85 % es un resultado excelente en este conjunto; superarla ampliamente debería levantar sospechas de fuga.

6. **Solo cuatro predictoras.** Con un espacio de variables tan pequeño y una muestra moderada, **los modelos simples son competitivos** y los complejos aportan poco. Es una buena ocasión para comprobar empíricamente que un bosque aleatorio no siempre supera a una regresión logística, y para valorar la **interpretabilidad** como criterio de elección.

7. **Los errores no cuestan lo mismo.** Un **falso negativo** —declarar benigna una masa maligna— retrasa el diagnóstico de un cáncer; un **falso positivo** conduce a una biopsia innecesaria, con su coste y su ansiedad asociada, pero no compromete la vida de la paciente. Aunque las clases estén equilibradas y la exactitud sea, por una vez, una métrica legítima, la evaluación clínica debe priorizar el **recall de la clase maligna** y ajustar el **umbral de decisión** en consecuencia.

## <font color="steelblue">Reglas del juego (buenas prácticas obligatorias)</font>

1. **Excluid `BI-RADS assessment` de las predictoras** (fuga: reproduce el juicio humano). Resérvadla aparte para el **benchmark** de la Fase 7.
2. **Codificad bien:** `Shape` y `Margin` con **one-hot**; `Density` como ordinal; **escalad `Age`** para modelos de distancia/lineales. Todo dentro de un **`Pipeline`** (sin fuga).
3. **Partición estratificada**; el *test* solo se toca al final.
4. **Coste asimétrico:** priorizad la **sensibilidad** (recall de *maligno*); elegid el **umbral** según el coste clínico, no el 0.5 por defecto.
5. **El equilibrado solo en *train***; aquí las clases están equilibradas, así que su efecto puede ser pequeño (es un resultado válido).
6. **Interpretabilidad:** el modelo debe poder **explicarse** (es un CAD clínico).
7. **Reproducibilidad y honestidad:** `random_state` fijado; reportad lo que no funcionó.

# <font color="steelblue">Fase 0 — Preparación del entorno y carga de datos</font>

In [ ]:
# !pip -q install kagglehub imbalanced-learn gradio optuna scikit-learn shap
import os, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
import kagglehub
RNG = 42

In [ ]:
# Descarga y carga (equivalente a usar %cd $path y leer el CSV)
path = kagglehub.dataset_download("overratedgman/mammographic-mass-data-set")
print("Ruta:", path, "| Archivos:", os.listdir(path))
mammogr = pd.read_csv(os.path.join(path, "Cleaned_data.csv"))
print(f"Dimensiones: {mammogr.shape[0]:,} filas × {mammogr.shape[1]} columnas")
mammogr.head()

# <font color="steelblue">Fase 1 — Comprensión y EDA</font>

**Tareas obligatorias**
1. **Columnas y tipos.** Comprobad los **nombres reales** de las columnas del CSV y mapeadlos a `BI-RADS`, `Age`, `Shape`, `Margin`, `Density`, `Severity`. Identificad qué es **nominal** (`Shape`, `Margin`) y qué **ordinal** (`Density`, `BI-RADS`).
2. **Objetivo.** Distribución de `Severity` (confirmad que está **equilibrado**).
3. **Faltantes.** ¿La versión "Cleaned" tiene `NaN`? ¿Cuántos había en el original (suele tener ~130 instancias con algún hueco)? Decidid una política.
4. **Relación con el objetivo.** Tasa de malignidad por `Shape`, `Margin`, `Density` y por **grupos de edad**. ¿Las formas **irregulares** y los márgenes **espiculados** se asocian a malignidad, como dice la clínica?
5. **Conclusión:** 3–4 hallazgos.

> **A responder:** dado el coste clínico, ¿qué métricas priorizaréis? (pista: **sensibilidad/recall de maligno**, especificidad, ROC-AUC y PR-AUC; la *accuracy* sola no basta).

# <font color="steelblue">Fase 2 — Preprocesado: exclusión de BI-RADS, codificación y partición</font>

**2A. Separar el objetivo y reservar el benchmark (obligatorio)**
1. `y = Severity`. **Quitad `BI-RADS assessment` de `X`** y **guardadla aparte** (la usaréis en la Fase 7 para comparar con el radiólogo).

**2B. Codificación correcta (clave)**
2. **`Shape` y `Margin` → one-hot** (son nominales; no impongáis orden).
3. **`Density` → ordinal** (mantener su orden). **`Age` → escalar** para logística/SVM/kNN (los árboles no lo necesitan).
4. Tratad los **faltantes** que queden (imputación dentro del `Pipeline`).

**2C. Partición y `Pipeline`**
5. **Partición estratificada** y `ColumnTransformer`/`Pipeline` que aplique cada transformación a su grupo de columnas.

> **A responder:** ¿por qué codificar `Shape`/`Margin` como enteros (1..5) puede empeorar y/o engañar a un modelo lineal o de distancia?

# <font color="steelblue">Fase 3 — Modelos base y comparación</font>

**Tareas obligatorias**
1. Comparad **≥5 familias** del curso: **Regresión logística**, **kNN**, **SVM**, **Árbol de decisión**, **Random Forest**, **HistGradientBoosting** (o XGBoost/LightGBM/CatBoost), **Naive Bayes**.
2. **Validación cruzada repetida** estratificada (el dataset es pequeño; aprovechadlo) con una métrica adecuada (**ROC-AUC** o **recall** de maligno; también F1).
3. **Tabla** comparativa y comentario. Incluid el **Árbol de decisión** y comentad su **interpretabilidad** (relevante en clínica).

# <font color="steelblue">Fase 4 — Ponderación de muestras y coste</font>

Las clases están **equilibradas**, así que el remuestreo puede aportar poco; aun así es obligatorio **medir** su efecto y, sobre todo, explorar la **ponderación por coste** (que aquí sí importa por la asimetría del error):

1. **Sin tratamiento** (línea base).
2. **Sensible al coste:** `class_weight` (o `sample_weight`) para **penalizar más** los falsos negativos (maligno clasificado como benigno).
3. (Opcional) **SMOTE**/submuestreo, para confirmar que con datos equilibrados aportan poco.

Reportad **recall de maligno**, **especificidad**, **F1** y ROC-AUC, y razonad la mejor opción **clínica**.

> **Lección esperable:** equilibrar un dataset ya equilibrado no suele mejorar; lo que sí cambia el comportamiento clínico es **ponderar el coste** y **mover el umbral** (Fase 7).

# <font color="steelblue">Fase 5 — Optimización de hiperparámetros</font>

1. Optimizad los **2–3 mejores** (modelo + ponderación).
2. `GridSearchCV`/`RandomizedSearchCV`/**Optuna**, con CV estratificada y la métrica elegida (ROC-AUC o recall); búsqueda **sobre el `Pipeline`** (prefijo `clf__`).
3. (Recomendado por el n pequeño) **CV anidada**.
4. Reportad mejores hiperparámetros y la mejora.

# <font color="steelblue">Fase 6 — Combinación de modelos</font>

1. Combinad los mejores con **`VotingClassifier`** (votación **blanda**) y/o **`StackingClassifier`**.
2. Comparad frente al **mejor individual**: ¿mejora ROC-AUC/recall? ¿compensa frente a la **pérdida de interpretabilidad** (importante en clínica)?
3. **Combinad solo si aporta** mejora real (requisito: *si fuera necesario*).

# <font color="steelblue">Fase 7 — Evaluación clínica, umbral y comparación con BI-RADS</font>

Esta es la fase **distintiva** del proyecto. El *test* se usa una sola vez.

**Tareas obligatorias**
1. **Métricas clínicas** del modelo final en el *test*: **matriz de confusión**, **sensibilidad (recall de maligno)**, **especificidad**, **F1**, **ROC-AUC** y **PR-AUC**.
2. **Umbral según coste.** No uséis 0.5: elegid el umbral con la **curva precision-recall**/ROC priorizando **sensibilidad** (minimizar malignos no detectados). Interpretad la **especificidad** como **"biopsias innecesarias evitadas"** y mostrad el compromiso.
3. **Comparación con el criterio del radiólogo (benchmark).** Con la columna **BI-RADS reservada**, definid una regla clínica simple (p. ej. **BI-RADS ≥ 4 ⇒ maligno**) y calculad **su** sensibilidad/especificidad sobre el **mismo *test***. **¿Vuestro modelo, sin usar BI-RADS, iguala o mejora al radiólogo?**
4. **Interpretabilidad:** árbol de decisión legible y/o **SHAP**. ¿Pesan forma irregular, margen espiculado y edad, como en la literatura?
5. **Discusión crítica:** un solo centro/época, implicaciones de un falso negativo, papel del CAD como **apoyo** (no sustituto).

# <font color="steelblue">Fase 8 — Despliegue del modelo (CAD)</font>

1. **Persistencia:** guardad el **`Pipeline` completo** con `joblib`.
2. **Función de predicción:** `recomendar(Age, Shape, Margin, Density)` que devuelva la **probabilidad de malignidad** y una **recomendación** según el umbral elegido (p. ej. *biopsia* vs *control a corto plazo*).
3. **Interfaz interactiva:** app con **Gradio** (o `ipywidgets`) con selectores para forma, margen, densidad y edad, que muestre probabilidad + recomendación. En Colab da un **enlace público** (incluidlo).
4. (Opcional, nota extra) **Streamlit**/**FastAPI**.

> **Aviso clínico (obligatorio en la interfaz):** herramienta **educativa** de apoyo a la decisión; **no** sustituye la valoración del radiólogo ni el diagnóstico histológico.

# <font color="steelblue">Pistas y errores típicos</font>

* **No uses BI-RADS como predictora.** Es el dictamen del radiólogo: el modelo "haría trampa". Resérvala para **comparar** tu modelo con el criterio clínico.
* **`Shape` y `Margin` son nominales.** Codifícalas con **one-hot**; tratarlas como 1..5 inventa un orden/distancia falsos (penaliza a modelos lineales y de distancia).
* **Equilibrado ≠ siempre útil.** Con clases parejas, remuestrear apenas cambia; lo que mueve la aguja clínica es **ponderar el coste** y **ajustar el umbral**.
* **Coste asimétrico:** un falso negativo (maligno no detectado) es lo peor; prioriza **sensibilidad** y explica el precio en especificidad (biopsias de más).
* **Interpretabilidad:** un árbol corto o reglas claras valen mucho en un CAD; acompáñalo de SHAP.
* **Despliegue:** guarda el **Pipeline entero** y respeta el **orden/formato** de las columnas (incluido el one-hot).

# <font color="steelblue">Referencias</font>

* Elter, M., Schulz-Wendtland, R. & Wittenberg, T. (2007). *The prediction of breast cancer biopsy outcomes using two CAD approaches that both emphasize an intelligible decision process*. Medical Physics, 34(11).
* Elter, M. (2007). *Mammographic Mass*. UCI ML Repository (id 161).
* American College of Radiology. *ACR BI-RADS Atlas*, 5th ed.
* *Using BI-RADS Descriptors and Ensemble Learning for Classifying Masses in Mammograms*. Springer LNCS, 2010.
* Cuadernos del curso: *Árboles de decisión*, *Random Forest*, *Boosting*, *Regresión logística binaria*, *Equilibrando las muestras*.
